In [ ]:
from src.models.reservoir_computing_model import IPReservoirComputingModel
from src.models.lstm_model import LSTMRegressorModel
from src.evaluation_framework.evalutaion_framework import plot_large_residual_histograms
from utils.logger import get_logger
from utils.io import read_csv_file
from src.data_extractor.data_preparer import FlightDataPreparer, ModelType

c:\Users\camar\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
logger = get_logger("Trainer")
DATA_SUFFIX = "extra_large"

train_df = read_csv_file(f"./data/datasets/opensky/preprocessed/train/data_{DATA_SUFFIX}.csv")
test_df = read_csv_file(f"./data/datasets/opensky/preprocessed/test/data_{DATA_SUFFIX}.csv")

display(train_df)
display(test_df)
data_preparer = FlightDataPreparer(target_column="velocity", window_size=12, required_length=100)

X_test_rc, y_test_rc, icao_list_test_rc = data_preparer.transform(test_df, model_type=ModelType.RESERVOIR_COMPUTING)
test_dataloader_lstm, n_features_test, icao_list_test_rc = data_preparer.transform(test_df, model_type=ModelType.LSTM)

Successfully loaded CSV. Shape: (2582149, 16)
Successfully loaded CSV. Shape: (674554, 16)


,lat,lat_diff,lon,lon_diff,velocity,velocity_diff,vertrate,baroaltitude,geoaltitude,heading,heading.1,time,onground,alert,spi,icao24
0,1.168126,-0.571589,0.540889,0.429390,-0.345972,2.698551,5.382430,-0.891759,-0.924420,0.632723,-0.774378,1641168010,0,0,0,3c4582.0
1,1.186520,0.194337,0.568716,-0.873867,0.093710,0.016494,0.127195,0.908575,0.822125,-0.966279,0.257497,1641168010,0,0,0,4ca8e8.0
2,0.560335,0.566561,0.549184,0.663540,0.924656,-0.033547,0.061504,1.053035,1.099781,0.790564,0.612380,1641168020,0,0,0,06a30c.0
3,-0.013736,0.208622,-0.584923,-0.013126,-1.713989,0.016494,-0.332638,-1.803663,-1.780674,-0.014924,0.999889,1641168020,0,0,0,a81032.0
4,0.059381,-0.017701,-0.635037,-0.013725,0.490365,0.268330,0.061504,0.619655,0.639409,-0.907072,0.420975,1641168020,0,0,0,a0d561.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2582144,0.289749,0.378331,-0.445830,-0.594097,0.257171,0.016494,-0.004186,0.762309,0.743306,-0.867934,0.496679,1641175190,0,0,0,ab2504.0
2582145,0.071046,-0.017701,-0.857761,-0.013725,-1.334621,0.016494,-0.858162,-1.287218,-1.282685,-0.205465,0.978664,1641175190,0,0,0,aacfbe.0
2582146,-0.581176,-0.017701,1.894293,-0.013725,0.474727,0.016494,0.981170,-0.424069,-0.365525,0.252422,0.967617,1641175190,0,0,0,781b94.0
2582147,0.261771,-0.017701,1.952430,-0.013725,-0.109982,0.016494,-0.923852,-0.127926,-0.157731,-0.987530,-0.157432,1641175190,0,0,0,781157.2


,lat,lat_diff,lon,lon_diff,velocity,velocity_diff,vertrate,baroaltitude,geoaltitude,heading,heading.1,time,onground,alert,spi,icao24
0,-3.361588,-0.328485,0.009383,-0.161574,-0.896305,0.016494,-0.726781,-1.258326,-1.241485,-0.553753,-0.832681,1641168020,0,0,0,e48003.0
1,0.454060,-0.427526,-0.308372,-0.176811,-1.379347,0.249827,-0.004186,-1.615865,-1.639159,-0.424720,-0.905325,1641168020,0,0,0,abea64.0
2,-3.360750,0.634767,0.008296,-0.410718,0.416075,-1.189942,-2.369042,-0.805082,-0.763200,-0.647648,0.761939,1641168020,0,0,0,e49329.0
3,-0.275715,0.615852,-0.567746,-0.362425,0.240645,0.076053,0.061504,0.621461,0.675236,-0.590270,0.807206,1641168020,0,0,0,abb3b6.0
4,0.201515,0.822308,-0.412050,-0.364704,0.875544,0.016494,0.061504,0.908575,0.922439,-0.454672,0.890659,1641168020,0,0,0,ad7760.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
674549,-4.177999,-0.184521,2.618967,-0.318507,-1.188282,1.690307,0.127195,-1.623088,-1.603333,-0.914013,-0.405685,1641175190,0,1,0,c818da.1
674550,-0.195837,-0.013761,-0.528321,-0.331439,-0.455227,0.016494,0.127195,0.908575,0.963640,-0.999978,0.006601,1641175190,0,0,0,ababaf.1
674551,-2.472143,-0.757192,0.121916,-0.319724,0.553386,0.016494,0.061504,0.908575,1.051415,-0.529179,-0.848510,1641175190,0,0,0,e07246.0
674552,-4.454172,-0.036452,2.613480,0.206661,-1.830128,0.592542,-0.069877,-1.411815,-1.397330,0.996428,-0.084443,1641175190,0,0,0,c805a0.2


In [ ]:
rc_model = IPReservoirComputingModel.load_model("hpt_rc", "1777152880")
y_pred_rc = rc_model.predict(X_test_rc)
lstm_model = LSTMRegressorModel.load_model("lstm_test", "1777200357")
y_true_lstm, y_pred_lstm = lstm_model.predict(test_dataloader_lstm)

Successfully loaded JSON as a dictionary.
Successfully loaded NPY from ./data/models\hpt_rc\1777152880\readout_weights.npy. Shape: (100, 1)


TypeError: 'NoneType' object cannot be interpreted as an integer